In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../train.csv")

X = df.drop(columns=["Price_INR_Lakhs", "Property_ID"])
y = df["Price_INR_Lakhs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

categorical_features = [
    "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)],
    sparse_threshold=0
)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

settings = [
    {"max_leaf_nodes":15},
    {"max_leaf_nodes":31},
    {"max_leaf_nodes":63}
]

results=[]

for setting in settings:
    model = Pipeline([
        ("preprocess",preprocessor),
        ("regressor",HistGradientBoostingRegressor(
            learning_rate=0.1,
            max_iter=100,
            max_leaf_nodes=setting["max_leaf_nodes"],
            random_state=42
        ))
    ])

    model.fit(X_train,y_train)

    y_pred = model.predict(X_val)

    regressor=model.named_steps["regressor"]

    

